# Tahap 2 CBR: Case Representation
### CBR Putusan Wanprestasi

**Tim:**
- Ahmad Qayyim — 202310370311286
- Bintang Mars Satria Tuhu — 202310370311410

---

## Tujuan

Mengubah teks bersih dari Tahap 1 menjadi case base terstruktur, dengan ekstraksi:
- **Metadata legal**: no_perkara, tanggal, pengadilan, pasal
- **Pihak**: nama penggugat dan tergugat (dengan anonimisasi)
- **Content**: ringkasan fakta, argumen hukum, amar putusan
- **Label**: kategori solusi (Dikabulkan / Dikabulkan Sebagian / Ditolak / NO)

Output:
- `data/processed/cases.csv`
- `data/processed/cases.json`

## 1. Setup

In [ ]:
!pip install pandas tqdm -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/kuliah/smt 6/Penalaran Komputer/Tugas SUB-CPMK 3/cbr-putusan-wanprestasi')
os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')

## 2. Verifikasi Input dari Tahap 1

Tahap 2 butuh hasil dari Tahap 1:
- `data/raw/metadata.json`
- `data/cleaned/case_*.txt`

In [ ]:
meta_path = PROJECT_DIR / 'data' / 'raw' / 'metadata.json'
clean_dir = PROJECT_DIR / 'data' / 'cleaned'

cleaned_files = sorted(clean_dir.glob('*.txt'))
print(f'metadata.json    : {"OK" if meta_path.exists() else "MISSING"}')
print(f'cleaned text files: {len(cleaned_files)} files')
if not meta_path.exists() or not cleaned_files:
    print('\nERROR: jalankan Tahap 1 dulu (notebook 01_case_base.ipynb)')

## 3. Run Case Representation

In [ ]:
import sys
import importlib.util

sys.path.insert(0, str(PROJECT_DIR / 'src'))

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

repr_mod = load_module('repr_mod', PROJECT_DIR / 'src' / '03_build_case_representation.py')
records  = repr_mod.run()
print(f'\nTotal cases ter-build: {len(records)}')

## 4. Inspeksi cases.csv

In [ ]:
import pandas as pd
import json

df = pd.read_csv('data/processed/cases.csv')
print(f'Shape: {df.shape}')
print(f'\nKolom: {list(df.columns)}')
df.head(3)

In [ ]:
# Sample 1 case lengkap
sample = df.iloc[0]
print(f'CASE ID         : {sample.case_id}')
print(f'No Perkara      : {sample.no_perkara}')
print(f'Tanggal Putusan : {sample.tanggal_putusan}')
print(f'Pengadilan      : {sample.pengadilan}')
print(f'Pasal           : {sample.pasal}')
print(f'Penggugat       : {sample.pihak_penggugat}')
print(f'Tergugat        : {sample.pihak_tergugat}')
print(f'Kategori Solusi : {sample.kategori_solusi}')
print(f'Word Count      : {sample.word_count}')
print(f'Quality         : {sample.extraction_quality}')
if sample.missing_fields:
    print(f'Missing fields  : {sample.missing_fields}')
print()
print('=== RINGKASAN FAKTA (preview 500 chars) ===')
print(str(sample.ringkasan_fakta)[:500])
print()
print('=== AMAR PUTUSAN (preview 500 chars) ===')
print(str(sample.amar_putusan)[:500])

## 5. Analisis Distribusi

In [ ]:
print('Distribusi extraction_quality:')
print(df['extraction_quality'].value_counts())
print()
print('Distribusi kategori_solusi:')
print(df['kategori_solusi'].value_counts())
print()
print('Statistik word_count:')
print(df['word_count'].describe())
print()
print('Statistik n_pasal:')
print(df['n_pasal'].describe())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 1. Kategori solusi
kat_counts = df['kategori_solusi'].value_counts()
axes[0, 0].bar(range(len(kat_counts)), kat_counts.values,
               color=['#2ecc71', '#3498db', '#e74c3c', '#f39c12', '#95a5a6'][:len(kat_counts)])
axes[0, 0].set_xticks(range(len(kat_counts)))
axes[0, 0].set_xticklabels(kat_counts.index, rotation=20, ha='right')
axes[0, 0].set_title('Distribusi Kategori Solusi')
axes[0, 0].set_ylabel('Jumlah')
for i, v in enumerate(kat_counts.values):
    axes[0, 0].text(i, v + 0.1, str(v), ha='center', fontweight='bold')

# 2. Distribusi per tahun
tahun_kat = df.groupby(['tahun', 'kategori_solusi']).size().unstack(fill_value=0)
tahun_kat.plot(kind='bar', stacked=True, ax=axes[0, 1], colormap='tab10')
axes[0, 1].set_title('Distribusi Solusi per Tahun')
axes[0, 1].set_xlabel('Tahun')
axes[0, 1].set_ylabel('Jumlah')
axes[0, 1].legend(title='Solusi', fontsize=8, loc='upper right')
axes[0, 1].tick_params(axis='x', rotation=0)

# 3. Word count distribution
axes[1, 0].hist(df['word_count'], bins=15, color='steelblue', edgecolor='white')
axes[1, 0].set_title('Distribusi Word Count')
axes[1, 0].set_xlabel('Jumlah kata')
axes[1, 0].set_ylabel('Frekuensi')
axes[1, 0].grid(True, alpha=0.3)

# 4. Extraction quality
qual_counts = df['extraction_quality'].value_counts()
colors_q = {'OK': '#2ecc71', 'PARTIAL': '#f39c12', 'FAILED': '#e74c3c'}
axes[1, 1].bar(qual_counts.index, qual_counts.values,
               color=[colors_q.get(q, 'gray') for q in qual_counts.index],
               edgecolor='black')
axes[1, 1].set_title('Extraction Quality')
axes[1, 1].set_ylabel('Jumlah')
for i, v in enumerate(qual_counts.values):
    axes[1, 1].text(i, v + 0.1, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('reports/figures/case_representation_stats.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Top Pasal yang Sering Disebutkan

In [ ]:
from collections import Counter

all_pasal = []
for p_str in df['pasal'].dropna():
    if p_str:
        all_pasal.extend([p.strip() for p in str(p_str).split(',') if p.strip()])

pasal_counter = Counter(all_pasal)
top_pasal = pasal_counter.most_common(15)

print('Top 15 pasal yang paling sering disebutkan:')
for pasal, cnt in top_pasal:
    print(f'  Pasal {pasal:30s}: {cnt} kasus')

## 7. Inspeksi Kasus dengan Extraction Bermasalah

In [ ]:
problematic = df[df['extraction_quality'].isin(['PARTIAL', 'FAILED'])]
if len(problematic) > 0:
    print(f'Kasus dengan masalah: {len(problematic)}')
    print(problematic[['case_id', 'no_perkara', 'extraction_quality', 'missing_fields']].to_string(index=False))
else:
    print('Semua kasus berhasil di-extract penuh.')

## 8. Final Checklist

In [ ]:
total = len(df)
ok_quality = (df['extraction_quality'] == 'OK').sum()
has_label  = (df['kategori_solusi'] != 'Tidak Teridentifikasi').sum()

print(f'Total cases       : {total}')
print(f'Quality OK        : {ok_quality} ({100*ok_quality/total:.1f}%)')
print(f'Berlabel solusi   : {has_label} ({100*has_label/total:.1f}%)')
print(f'cases.csv         : {Path("data/processed/cases.csv").exists()}')
print(f'cases.json        : {Path("data/processed/cases.json").exists()}')
print()

if total >= 30 and has_label >= 25:
    print('STATUS: SIAP lanjut ke Tahap 3 (Case Retrieval)')
else:
    print('STATUS: Cek kualitas — beberapa case mungkin perlu review manual.')